# Saxo Institutional Dashboard (Presentation)

Read-only Databricks notebook for the first institutional page on the live Saxo stack.
No workflow, no schedule, and no production writes are performed here.


### Source tables
- `pawan_quant_research_1.v4.gold_rt_saxo_dashboard_feed`
- `pawan_quant_research_1.v4.gold_rt_saxo_dashboard_health_rollup`
- `pawan_quant_research_1.v4.gold_rt_saxo_dashboard_performance_rollup`
- `pawan_quant_research_1.v4.bronze_rt_regime_signal`
- `pawan_quant_research_1.v4.bronze_rt_saxo_trade_log`


In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

CATALOG = "pawan_quant_research_1"
SCHEMA = "v4"
FEED_TABLE = f"{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_feed"
HEALTH_TABLE = f"{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_health_rollup"
PERF_TABLE = f"{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_performance_rollup"
REGIME_SIGNAL_TABLE = f"{CATALOG}.{SCHEMA}.bronze_rt_regime_signal"
TRADE_LOG_TABLE = f"{CATALOG}.{SCHEMA}.bronze_rt_saxo_trade_log"

pd.set_option("display.max_columns", 200)


def is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
    except Exception:
        return False
    if isinstance(missing, (bool, int, float)):
        return bool(missing)
    return False


def as_float(value):
    try:
        if is_missing(value):
            return None
        return float(value)
    except Exception:
        return None


def as_int(value):
    try:
        if is_missing(value):
            return None
        return int(float(value))
    except Exception:
        return None


def as_text(value, default="N/A"):
    if is_missing(value):
        return default
    return str(value)


def fmt_money(value):
    v = as_float(value)
    if v is None:
        return "N/A"
    return f"${v:,.2f}"


def fmt_pct(value):
    v = as_float(value)
    if v is None:
        return "N/A"
    return f"{v:+,.2%}" if v < 0 else f"{v:,.2%}"


def fmt_minutes(value):
    v = as_float(value)
    if v is None:
        return "N/A"
    return f"{v:.0f}"


def fmt_timestamp(value):
    ts = pd.to_datetime(value, errors="coerce")
    if pd.isna(ts):
        return "N/A"
    return ts.strftime("%Y-%m-%d %H:%M:%S")


def spark_to_pandas(query: str, force_no_arrow: bool = True):
    """Convert Spark SQL to pandas with deterministic typing behavior."""
    if not force_no_arrow:
        return spark.sql(query).toPandas()

    arrow_key = "spark.sql.execution.arrow.pyspark.enabled"
    prev_arrow = None
    try:
        prev_arrow = spark.conf.get(arrow_key)
    except Exception:
        prev_arrow = None

    try:
        spark.conf.set(arrow_key, "false")
        return spark.sql(query).toPandas()
    finally:
        if prev_arrow is None:
            try:
                spark.conf.unset(arrow_key)
            except Exception:
                spark.conf.set(arrow_key, "true")
        else:
            try:
                spark.conf.set(arrow_key, prev_arrow)
            except Exception:
                pass


def empty_row():
    return pd.Series(dtype="object")


def query_to_pandas(query: str, empty_columns=None):
    try:
        return spark_to_pandas(query)
    except Exception as exc:
        print(f"Query failed; continuing with empty result: {exc}")
        return pd.DataFrame(columns=empty_columns or [])


def latest_row(table_name, order_col, where_clause="", required=False):
    q = f"SELECT * FROM {table_name} {where_clause} ORDER BY {order_col} DESC LIMIT 1"
    rows = query_to_pandas(q)
    if rows.empty:
        if required:
            raise ValueError(f"No rows in {table_name}")
        print(f"No rows found in {table_name}; continuing with empty values.")
        return empty_row()
    return rows.iloc[0]


def latest_rows(table_name, order_col, limit=360):
    q = (
        f"SELECT * FROM ("
        f" SELECT * FROM {table_name}"
        f" ORDER BY {order_col} DESC LIMIT {int(limit)}"
        f") t ORDER BY {order_col} ASC"
    )
    return query_to_pandas(q)


def row_fields_frame(row, fields):
    records = []
    for field in fields:
        raw_value = row.get(field) if hasattr(row, "get") else None
        display_value = "" if is_missing(raw_value) else str(raw_value)
        records.append((field, display_value))
    return pd.DataFrame(records, columns=["field", "value"])


def sql_literal_timestamp(ts):
    if pd.isna(ts):
        return "NULL"
    return ts.strftime("%Y-%m-%d %H:%M:%S")


## Executive KPI strip


In [ ]:
latest_feed = latest_row(FEED_TABLE, "feed_ts")
latest_health = latest_row(HEALTH_TABLE, "rollup_ts")
latest_perf = latest_row(PERF_TABLE, "rollup_ts")

kpis = pd.DataFrame(
    [
        ("Current regime", as_text(latest_feed.get("regime_name", "N/A"))),
        ("Account equity", fmt_money(latest_feed.get("account_equity"))),
        ("Current position", as_text(latest_feed.get("actual_ticker", latest_feed.get("position", "N/A")))),
        ("Execution mode", as_text(latest_feed.get("execution_mode", latest_perf.get("execution_mode", "N/A")))),
        ("Alpha vs SPY", fmt_pct(latest_feed.get("alpha_pct"))),
        ("Drawdown", fmt_pct(latest_feed.get("drawdown_pct"))),
        ("Freshness (min)", fmt_minutes(latest_feed.get("dashboard_data_freshness_minutes"))),
        ("Last execution action", as_text(latest_feed.get("last_execution_action", "N/A"))),
        ("1h strategy", fmt_pct(latest_perf.get("strategy_return_1h"))),
        ("1h alpha", fmt_pct(latest_perf.get("alpha_1h"))),
        ("1d strategy", fmt_pct(latest_perf.get("strategy_return_1d"))),
        ("1d alpha", fmt_pct(latest_perf.get("alpha_1d"))),
        ("Data timestamp", fmt_timestamp(latest_feed.get("feed_ts"))),
    ],
    columns=["Metric", "Value"],
)
display(kpis)


## Strategy vs SPY with regime shading


In [ ]:
MAX_HISTORY_POINTS = 360

execution_mode = latest_feed.get("execution_mode")
if execution_mode is None:
    execution_mode = latest_perf.get("execution_mode")
mode_filter = ""
mode_value = as_text(execution_mode, default="")
if mode_value and mode_value != "N/A":
    mode_filter = "WHERE execution_mode = '" + mode_value.replace("'", "''") + "'"

# Prefer performance rollup history, but fall back to trade-log equity history if the rollup is not ready yet.
history = latest_rows(PERF_TABLE, "rollup_ts", limit=MAX_HISTORY_POINTS)
history_label = "performance rollup"
if history.empty or "rollup_ts" not in history.columns or "account_equity" not in history.columns:
    history = query_to_pandas(
        f"""
        SELECT trade_ts AS rollup_ts, account_equity, execution_mode, spy_close
        FROM {TRADE_LOG_TABLE}
        {mode_filter}
        ORDER BY trade_ts DESC
        LIMIT {int(MAX_HISTORY_POINTS)}
        """,
        empty_columns=["rollup_ts", "account_equity", "execution_mode", "spy_close"],
    )
    if not history.empty:
        history = history.sort_values("rollup_ts").reset_index(drop=True)
    history_label = "trade log fallback"

if history.empty and latest_feed.get("feed_ts") is not None and latest_feed.get("account_equity") is not None:
    history = pd.DataFrame(
        [{"rollup_ts": latest_feed.get("feed_ts"), "account_equity": latest_feed.get("account_equity")}]
    )
    history_label = "latest feed snapshot"

history_available = not history.empty and "rollup_ts" in history.columns and "account_equity" in history.columns
if history_available:
    history = history.sort_values("rollup_ts").reset_index(drop=True)
    history["rollup_ts"] = pd.to_datetime(history["rollup_ts"], errors="coerce")
    history["account_equity"] = pd.to_numeric(history["account_equity"], errors="coerce")
    if "spy_close" in history.columns:
        history["spy_close"] = pd.to_numeric(history["spy_close"], errors="coerce")
    history = history.dropna(subset=["rollup_ts", "account_equity"]).reset_index(drop=True)
    history_available = not history.empty

if not history_available:
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.axis("off")
    ax.text(0.5, 0.5, "No dashboard performance history available yet.", ha="center", va="center")
    plt.tight_layout()
    plt.show()
else:
    window_start = history["rollup_ts"].min()
    window_end = history["rollup_ts"].max()
    window_start_s = sql_literal_timestamp(window_start)
    window_end_s = sql_literal_timestamp(window_end)

    spy_history = pd.DataFrame(columns=["trade_ts", "spy_close"])
    if "spy_close" in history.columns and history["spy_close"].notna().any():
        spy_history = history[["rollup_ts", "spy_close"]].rename(columns={"rollup_ts": "trade_ts"}).copy()
    else:
        escaped_mode_value = mode_value.replace("'", "''") if mode_value and mode_value != "N/A" else ""
        mode_clause = f"AND execution_mode = '{escaped_mode_value}'" if escaped_mode_value else ""
        spy_history = query_to_pandas(
            f"""
            SELECT trade_ts, spy_close
            FROM {TRADE_LOG_TABLE}
            WHERE trade_ts BETWEEN CAST('{window_start_s}' AS TIMESTAMP)
              AND CAST('{window_end_s}' AS TIMESTAMP)
              AND spy_close IS NOT NULL
              {mode_clause}
            ORDER BY trade_ts ASC
            """,
            empty_columns=["trade_ts", "spy_close"],
        )
        if spy_history.empty and mode_clause:
            spy_history = query_to_pandas(
                f"""
                SELECT trade_ts, spy_close
                FROM {TRADE_LOG_TABLE}
                WHERE trade_ts BETWEEN CAST('{window_start_s}' AS TIMESTAMP)
                  AND CAST('{window_end_s}' AS TIMESTAMP)
                  AND spy_close IS NOT NULL
                ORDER BY trade_ts ASC
                """,
                empty_columns=["trade_ts", "spy_close"],
            )
        if spy_history.empty:
            spy_history = query_to_pandas(
                f"""
                SELECT trade_ts, spy_close
                FROM {TRADE_LOG_TABLE}
                WHERE spy_close IS NOT NULL
                ORDER BY trade_ts DESC
                LIMIT 500
                """,
                empty_columns=["trade_ts", "spy_close"],
            )
            if not spy_history.empty:
                spy_history = spy_history.sort_values("trade_ts")

    if not spy_history.empty:
        spy_history["trade_ts"] = pd.to_datetime(spy_history["trade_ts"], errors="coerce")
        spy_history["spy_close"] = pd.to_numeric(spy_history["spy_close"], errors="coerce")
        spy_history = spy_history.dropna(subset=["trade_ts", "spy_close"]).sort_values("trade_ts").reset_index(drop=True)

    aligned = history[["rollup_ts", "account_equity"]].copy()
    if not spy_history.empty:
        aligned = pd.merge_asof(
            aligned.sort_values("rollup_ts").reset_index(drop=True),
            spy_history[["trade_ts", "spy_close"]].sort_values("trade_ts").reset_index(drop=True),
            left_on="rollup_ts",
            right_on="trade_ts",
            direction="backward",
            allow_exact_matches=True,
        )
        aligned = aligned.drop(columns=["trade_ts"])
    else:
        aligned["spy_close"] = pd.NA

    aligned["account_equity"] = pd.to_numeric(aligned["account_equity"], errors="coerce")
    aligned = aligned[pd.notna(aligned["account_equity"])].dropna(subset=["rollup_ts"]).reset_index(drop=True)

    regime_table = query_to_pandas(
        f"""
        WITH prior AS (
            SELECT CAST(timestamp AS TIMESTAMP) AS regime_ts,
                   COALESCE(regime_name, CAST(regime_int AS STRING)) AS regime_name,
                   CAST(regime_int AS INT) AS regime_int
            FROM {REGIME_SIGNAL_TABLE}
            WHERE CAST(timestamp AS TIMESTAMP) < CAST('{window_start_s}' AS TIMESTAMP)
            ORDER BY CAST(timestamp AS TIMESTAMP) DESC
            LIMIT 1
        ),
        windowed AS (
            SELECT CAST(timestamp AS TIMESTAMP) AS regime_ts,
                   COALESCE(regime_name, CAST(regime_int AS STRING)) AS regime_name,
                   CAST(regime_int AS INT) AS regime_int
            FROM {REGIME_SIGNAL_TABLE}
            WHERE CAST(timestamp AS TIMESTAMP) >= CAST('{window_start_s}' AS TIMESTAMP)
              AND CAST(timestamp AS TIMESTAMP) <= CAST('{window_end_s}' AS TIMESTAMP)
            ORDER BY CAST(timestamp AS TIMESTAMP) ASC
        )
        SELECT * FROM (
            SELECT * FROM prior
            UNION ALL
            SELECT * FROM windowed
        )
        ORDER BY regime_ts ASC
        """,
        empty_columns=["regime_ts", "regime_name", "regime_int"],
    )

    if "regime_ts" in regime_table.columns:
        regime_table["regime_ts"] = pd.to_datetime(regime_table["regime_ts"], errors="coerce")
        regime_table = regime_table.dropna(subset=["regime_ts"]).sort_values("regime_ts").reset_index(drop=True)
        regime_table = regime_table[(regime_table["regime_ts"] >= window_start) & (regime_table["regime_ts"] <= window_end)]
    else:
        regime_table = pd.DataFrame(columns=["regime_ts", "regime_name", "regime_int"])

    color_by_regime = {
        "Expansion": "#2E8B57",
        "Slowdown": "#F5A623",
        "Recession": "#C0392B",
    }

    fig, (ax_perf, ax_regime) = plt.subplots(
        2,
        1,
        figsize=(16, 8),
        sharex=True,
        gridspec_kw={"height_ratios": [4, 1]},
    )

    if aligned.empty:
        ax_perf.axis("off")
        ax_perf.text(0.5, 0.5, "No aligned equity history available.", ha="center", va="center")
    else:
        strategy_base = aligned.loc[aligned.index[0], "account_equity"]
        aligned["strategy_return"] = aligned["account_equity"] / strategy_base - 1.0
        spy_base_candidates = aligned.loc[aligned["spy_close"].notna(), "spy_close"]
        if not spy_base_candidates.empty:
            spy_base = float(spy_base_candidates.iloc[0])
            aligned["spy_return"] = aligned["spy_close"].astype(float) / spy_base - 1.0
        else:
            aligned["spy_return"] = float("nan")

        ax_perf.plot(aligned["rollup_ts"], aligned["strategy_return"] * 100, linewidth=2, label="Strategy")
        if aligned["spy_return"].notna().any():
            ax_perf.plot(aligned["rollup_ts"], aligned["spy_return"] * 100, linewidth=1.7, linestyle="--", label="SPY")
        ax_perf.set_title(f"Saxo strategy vs SPY ({history_label})")
        ax_perf.set_ylabel("Return (%)")
        ax_perf.grid(alpha=0.25)
        ax_perf.legend(loc="upper left")

    if len(regime_table) == 0:
        print("No regime transitions found for the selected history window; rendering without regime shading.")
    else:
        for idx, row in regime_table.iterrows():
            regime_start = row["regime_ts"]
            regime_end = regime_table.loc[idx + 1, "regime_ts"] if idx + 1 < len(regime_table) else window_end
            regime_start = max(regime_start, window_start)
            regime_end = min(regime_end, window_end)
            if pd.isna(regime_start) or pd.isna(regime_end) or regime_end <= regime_start:
                continue
            color = color_by_regime.get(as_text(row.get("regime_name"), default="Unknown"), "#9E9E9E")
            ax_regime.axvspan(regime_start, regime_end, color=color, alpha=0.16)

    ax_regime.set_ylim(0, 1)
    ax_regime.set_yticks([])
    ax_regime.set_ylabel("Regime")
    ax_regime.set_xlabel("Timestamp")
    ax_regime.legend(
        handles=[
            plt.Line2D([0], [0], color="#2E8B57", lw=6, alpha=0.25),
            plt.Line2D([0], [0], color="#F5A623", lw=6, alpha=0.25),
            plt.Line2D([0], [0], color="#C0392B", lw=6, alpha=0.25),
        ],
        labels=["Expansion", "Slowdown", "Recession"],
        loc="upper left",
    )

    plt.tight_layout()
    plt.show()


## Portfolio / position


In [ ]:
portfolio_cols = [
    "position",
    "target_ticker",
    "actual_ticker",
    "account_cash",
    "positions_count",
    "positions_market_value",
    "positions_unrealized_plpc",
    "equity_vs_cash_ratio",
    "regime_age_hours",
]
display(row_fields_frame(latest_feed, portfolio_cols))

positions_json = latest_feed.get("positions_json")
if isinstance(positions_json, str) and positions_json.strip():
    try:
        displayHTML("<pre>" + json.dumps(json.loads(positions_json), indent=2) + "</pre>")
    except Exception:
        displayHTML(f"<pre>{positions_json}</pre>")


## Risk / drawdown


In [ ]:
risk_rows = [
    ("Current drawdown", fmt_pct(latest_feed.get("drawdown_pct"))),
    ("Regime drawdown", fmt_pct(latest_feed.get("regime_drawdown_pct"))),
    ("Total return", fmt_pct(latest_feed.get("total_return_pct"))),
    ("Regime return", fmt_pct(latest_feed.get("regime_return_pct"))),
    ("Alpha", fmt_pct(latest_feed.get("alpha_pct"))),
    ("Peak equity", fmt_money(latest_feed.get("peak_equity"))),
    ("Regime peak equity", fmt_money(latest_feed.get("regime_peak_equity"))),
]

display(pd.DataFrame(risk_rows, columns=["Metric", "Value"]))


## Freshness / execution health


In [ ]:
health_rows = [
    ("Dashboard freshness (min)", fmt_minutes(latest_feed.get("dashboard_data_freshness_minutes"))),
    ("Audit lag (min)", fmt_minutes(latest_health.get("audit_lag_minutes"))),
    ("Latest audit ts", fmt_timestamp(latest_feed.get("latest_audit_ts"))),
    ("Execution action", as_text(latest_feed.get("last_execution_action", "N/A"))),
    ("Skip reason", as_text(latest_feed.get("last_skip_reason", "N/A"))),
    ("Error reason", as_text(latest_feed.get("last_error_reason", "N/A"))),
    ("Skip / 24h", as_text(latest_feed.get("last_skip_count_24h", 0))),
    ("Error / 24h", as_text(latest_feed.get("last_error_count_24h", 0))),
    ("Rate limits / 24h", as_text(latest_feed.get("last_rate_limit_count_24h", 0))),
    ("Regime transitions / 24h", as_text(latest_health.get("regime_transition_count_24h", 0))),
]

display(pd.DataFrame(health_rows, columns=["Metric", "Value"]))


### Optional drilldown context


In [ ]:
regime_drilldown = query_to_pandas(
    f"""
    SELECT timestamp,
        COALESCE(regime_name, CAST(regime_int AS STRING)) AS regime_name,
        position
    FROM {REGIME_SIGNAL_TABLE}
    ORDER BY timestamp DESC
    LIMIT 20
    """
)

if regime_drilldown.empty or "timestamp" not in regime_drilldown.columns:
    display(pd.DataFrame(columns=["timestamp", "regime_name", "position"]))
else:
    regime_drilldown["timestamp"] = pd.to_datetime(regime_drilldown["timestamp"], errors="coerce")
    regime_drilldown["timestamp"] = regime_drilldown["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
    display(regime_drilldown)
